# ステップ4b：南極を細かく見る（氷採掘・有人で「南極」を候補にした班）

このノートブックは `course_moonbase.ipynb` の**分岐**です。

- **氷採掘**を選んだ班
- **太陽光発電・有人総合**で、ステップ4で「南極」を候補地にした班

だけが使います。電波天文台の班や、赤道・裏側を選んだ班は**このノートブックは不要**――
`course_moonbase.ipynb` のステップ5に進んでください。

南極には「一年中日が当たらない場所（永久影）」と「ほぼずっと日が当たる場所」がとなり合い、
地面の傾き（傾斜）も場所で大きく違います。永久影は `load('環境')` には入っていないので、
ここで LOLA の極域専用データ（日照率・傾斜・永久影率）を使います。

In [ ]:
# 準備：ヘルパー（moonkit）を読み込む
from moonkit import *

MY_MISSION = '氷採掘'   # ★ここを変える：'氷採掘' / '有人総合' / '太陽光発電'
print('このノートブックは氷採掘・有人・（南極を候補にした）太陽光の班向けです。')
print('選んだミッション：', MY_MISSION)

**予想**：南極で「よく日が当たる場所」と「平らな場所」は、同じ場所にあると思う？

In [ ]:
極 = south_pole(load('極域日照'))          # 南緯80度より南（日照率・永久影率・傾斜 slope_deg）
my_threshold = 5                          # ★ここを変える：「これより日照率が低ければ永久影だろう」[%]

hist(極, 'average_illumination_percent', vline=my_threshold)
暗い場所 = 極[極['average_illumination_percent'] <= my_threshold]
print('しきい値', my_threshold, '% 以下の地点：', len(暗い場所), '個')

my_slope_max = 8                          # ★ここを変える：「これより急だと基地に向かない」[度]
hist(極, 'slope_deg', vline=my_slope_max)
平ら = 極[極['slope_deg'] <= my_slope_max]
print('傾斜', my_slope_max, '度以下（平ら）の地点：', len(平ら), '個 /', len(極))

両方 = 極[(極['average_illumination_percent'] >= 30) & (極['slope_deg'] <= my_slope_max)]
print('日照30%以上 かつ 平ら：', len(両方), '個')

In [ ]:
# 南極の各地点に「いちばん近い永久影までの距離（km_to_shadow）」を付けてスコア式で評価
極 = dist_to_permanent_shadow(south_pole(load('極域日照')))

want_pole = {                                          # ★ここを変える：氷採掘なら 永久影まで を重く
    'average_illumination_percent': ('高い', 3),
    'slope_deg':                    ('低い', 2),
    'km_to_shadow':                 ('低い', 0),
    'permanent_shadow_fraction':    ('低い', 0),
}
best_pole = site_score(極, want_pole, top=10)
print(best_pole[['lat', 'lon', 'average_illumination_percent',
                 'slope_deg', 'km_to_shadow', 'スコア']].round(2).to_string(index=False))
scatter(site_score(極, want_pole, top=None), 'lon', 'lat', color='スコア')

**ワークシートに記録**：日照・傾斜のしきい値と当てはまった地点数。スコア上位の緯度経度・日照率・氷までの距離。

**気づいたこと**：いちばん平らな地点は、日が当たっている？　日照を重くすると傾斜は？

---
## 終わったら

`course_moonbase.ipynb` の**ステップ5**に戻り、他の班（ちがうミッション）と結果を持ち寄ります。
南極という結論は「氷採掘・有人にとっては正当」ですが、**電波天文＝裏側・通信＝表側**のように
ミッションが変われば行き先も変わります。そこを共有するのがステップ5です。